# Cloud Warehouses, Cost Control and the Cloud Data Services Map

Notebook for **Chapter 8** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch08.html

Tip: change a number, predict the output, then run the cell again.

## 8.1 What Cloud Warehouses Have in Common (and Where They Differ)

## 8.2 Partitioning, Clustering and Scan Pruning

In [ ]:
import random
random.seed(7)
days=365; rows_per_day=10000; bytes_per_row=100
countries=["IN","US","UK","DE","SG"]
data=[(d,random.choice(countries)) for d in range(days) for _ in range(rows_per_day)]
total=len(data)*bytes_per_row
print(f"table: {len(data):,} rows, {total/1e6:.0f} MB")
print("no partition, WHERE day=100 :", f"{total/1e6:.0f} MB scanned")
print("partition by day, day=100   :", f"{rows_per_day*bytes_per_row/1e6:.0f} MB scanned")
print("partition by day, 30 days   :", f"{30*rows_per_day*bytes_per_row/1e6:.0f} MB scanned")
day=[c for d,c in data if d==100]
def blocks_read(rows,target,bs=1000):
    return sum(1 for i in range(0,len(rows),bs) if target in rows[i:i+bs])
print("unsorted day: blocks containing 'DE':",blocks_read(day,"DE"),"of",len(day)//1000)
print("sorted  day: blocks containing 'DE':",blocks_read(sorted(day),"DE"),"of",len(day)//1000)
rate=5.0   # illustrative price per TB scanned, NOT a real price
for lab,mb in (("full scan",total/1e6),("1 day",rows_per_day*bytes_per_row/1e6)):
    print(f"{lab}: {mb:.0f} MB -> cost at {rate}/TB = {mb/1e6*rate:.6f} units; x1000 queries/day = {mb/1e6*rate*1000:.2f}")

## 8.3 The Cloud Services Map and the Non-SQL Stores

## 8.4 Cost Arithmetic and Tuning: BigQuery, Snowflake and Redshift Worked Through

The sections above gave you the vocabulary. This one gives you the habit that separates a junior from an expert: before you run a query, estimate what it will cost, and after you run it, check that the estimate was right. Every warehouse bills for one of two things. **Scan-based** engines (BigQuery on-demand, Athena) charge for bytes read. **Time-based** engines (Snowflake, Redshift provisioned) charge for how long a cluster is awake. The tuning levers differ, but the question is the same: how do I make the engine touch less data, or stay awake for less time?

In [ ]:
# BigQuery-style on-demand cost: you pay for bytes scanned, so layout and columns matter.
TB = 1024**4
table_bytes = 2 * TB           # 2 TiB fact table, 365 daily partitions, 40 columns
price_per_tib = 6.25           # illustrative USD per TiB scanned, check current pricing
col_share = {"order_id": 0.02, "order_date": 0.01, "amount": 0.02}   # share of table bytes per column
def cost(b): return b / TB * price_per_tib
scan_all = table_bytes
scan_cols = table_bytes * sum(col_share.values())
days = 7
scan_cols_part = scan_cols * days / 365
for name, b in [("SELECT * , no filter", scan_all),
                ("3 columns, no filter", scan_cols),
                ("3 columns, 7 of 365 partitions", scan_cols_part)]:
    print(f"{name:34s} {b/1024**3:9.2f} GiB  ${cost(b):8.4f}")
print("saving, first vs last:", round(scan_all / scan_cols_part), "x")
print("30 dashboard refreshes a day for a month, last query: $", round(cost(scan_cols_part) * 30 * 30, 2))
print("same refreshes with SELECT *: $", round(cost(scan_all) * 30 * 30, 2))

SELECT * , no filter                 2048.00 GiB  $ 12.5000
3 columns, no filter                  102.40 GiB  $  0.6250
3 columns, 7 of 365 partitions          1.96 GiB  $  0.0120
saving, first vs last: 1043 x
30 dashboard refreshes a day for a month, last query: $ 10.79
same refreshes with SELECT *: $ 11250.0


In [ ]:
# Snowflake-style credit arithmetic. Warehouse size doubles credits per hour at each step.
credits_per_hour = {"XS": 1, "S": 2, "M": 4, "L": 8, "XL": 16}
price_per_credit = 3.0   # illustrative USD
def run_cost(size, minutes, billed_min=1.0):
    return credits_per_hour[size] * max(minutes, billed_min) / 60 * price_per_credit
# an ETL job: 40 min on M. On L it often finishes in a bit over half the time, not exactly half.
for size, minutes in [("M", 40), ("L", 22), ("XL", 13), ("XL", 20)]:
    print(f"{size:3s} {minutes:3d} min  credits={credits_per_hour[size]*minutes/60:5.2f}  cost=${run_cost(size, minutes):5.2f}")
# idle cost: dashboards hit an M warehouse for 5 seconds every 10 minutes.
def day_cost(auto_suspend_s):
    runs = 24 * 6                       # one query every 10 min
    # each query keeps the warehouse awake for 5 s of work + the auto-suspend wait,
    # capped at the 600 s gap between queries; every resume bills at least 60 s
    awake = min(5 + auto_suspend_s, 600)
    awake = max(awake, 60)
    return runs * awake / 3600 * credits_per_hour["M"] * price_per_credit
for s in (600, 300, 60):
    print(f"auto-suspend {s:3d}s -> ${day_cost(s):6.2f} per day")

M    40 min  credits= 2.67  cost=$ 8.00
L    22 min  credits= 2.93  cost=$ 8.80
XL   13 min  credits= 3.47  cost=$10.40
XL   20 min  credits= 5.33  cost=$16.00
auto-suspend 600s -> $288.00 per day
auto-suspend 300s -> $146.40 per day
auto-suspend  60s -> $ 31.20 per day


In [ ]:
# Zone maps (Redshift sort keys, Snowflake micro-partitions, BigQuery clustering all use the idea):
# each block stores min and max of a column; a filter skips blocks whose range cannot match.
import random
random.seed(7)
N, BLOCK = 100_000, 1_000                      # 100 blocks of 1,000 rows
dates = [random.randint(0, 364) for _ in range(N)]
def blocks_scanned(vals, lo, hi):
    n = 0
    for i in range(0, len(vals), BLOCK):
        blk = vals[i:i+BLOCK]
        if max(blk) >= lo and min(blk) <= hi: n += 1
    return n
lo, hi = 100, 106                               # a one week filter
unsorted_ = blocks_scanned(dates, lo, hi)
sorted_ = blocks_scanned(sorted(dates), lo, hi)
print("rows matching:", sum(lo <= d <= hi for d in dates))
print("blocks scanned, unsorted:", unsorted_, "of", N // BLOCK)
print("blocks scanned, sorted by date:", sorted_, "of", N // BLOCK)
# second filter column: customer id, sorted by date only. Does it help? no.
cust = [random.randint(1, 5000) for _ in range(N)]
order = sorted(range(N), key=lambda i: dates[i])
cust_by_date = [cust[i] for i in order]
print("customer filter after sorting by date:", blocks_scanned(cust_by_date, 42, 42), "of 100 blocks")

rows matching: 1936
blocks scanned, unsorted: 100 of 100
blocks scanned, sorted by date: 3 of 100
customer filter after sorting by date: 100 of 100 blocks


## Chapter recap

## Practice set

**1.** (easy) Name two things you can do to reduce the cost of a time-billed warehouse. **Answer:** auto-suspend when idle and right-size the cluster (also stop duplicate dashboards from re-running heavy queries).

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch08.html